# 03 — Pre-process 3DSES Gold (T1.8) and check the class mapping (T1.5)

What it does (CPU only — **do not switch the GPU on for this notebook**):

1. downloads Gold + test_area (about 4 GB) from the Hugging Face mirror,
2. shifts coordinates by one fixed offset so they fit in float32 without losing millimetres,
3. voxel-subsamples every station at **2 cm** and **1 cm** (one real point kept per voxel, with its own colour, intensity and labels),
4. stores, for every station, the map *original point → kept point*, so predictions can be copied back to every original point (needed for the Codabench submission and for full-resolution validation),
5. writes per-class statistics before / after subsampling,
6. checks the class-ID → name mapping with a height-above-floor table and two pictures.

Output folder `/kaggle/working/3dses_gold_v1/` — afterwards turn it into a **private Kaggle Dataset** (Output tab → "New Dataset") and add the other members as collaborators.

In [ ]:
import os, sys, json, time, subprocess
import numpy as np

ON_KAGGLE = os.path.isdir("/kaggle/working")
RAW = "/kaggle/temp/3dses" if ON_KAGGLE else "./data/3dses"
OUT = "/kaggle/working/3dses_gold_v1" if ON_KAGGLE else "./results/3dses_gold_v1"
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)

VOXELS = {"2cm": 0.02, "1cm": 0.01}
OFFSET = np.array([900.0, 480.0, 100.0])     # one fixed shift for all stations (survey frame -> small numbers)
TRAIN = ["S163", "S164", "S165", "S168", "S169", "S178", "S179"]   # Gold stations with manual labels
TEST = ["S170", "S171", "S180"]                                     # official test area, no labels
CLASS_NAMES = ["Column", "Components", "Covering", "Damper", "Door", "Exit sign", "Fire terminal", "Furniture",
               "Heater", "Lamp", "Outlet", "Railing", "Slab", "Stair", "Switch", "Wall", "Window", "Clutter"]
SMALL = [3, 5, 6, 10, 14]                    # Damper, Exit sign, Fire terminal, Outlet, Switch (proposal)
print("RAW:", RAW, "| OUT:", OUT)

In [ ]:
try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import snapshot_download

have = all(os.path.isfile(os.path.join(RAW, "Gold", s + ".npy")) for s in TRAIN) and \
       all(os.path.isfile(os.path.join(RAW, "test_area", s + ".npy")) for s in TEST)
if not have:
    t0 = time.time()
    snapshot_download(repo_id="heig-vd-geo/3DSES", repo_type="dataset",
                      allow_patterns=["Gold/*", "test_area/*"], local_dir=RAW)
    print(f"downloaded in {(time.time() - t0) / 60:.1f} min")
print("Gold:", sorted(os.listdir(os.path.join(RAW, "Gold"))))
print("test:", sorted(os.listdir(os.path.join(RAW, "test_area"))))

## Voxel subsampling

For each voxel the point **closest to the voxel centre** is kept (a real measured point, not an average), with its own label.
`inverse[i]` gives, for original point `i`, the index of the kept point of its voxel.

In [ ]:
def voxel_subsample(xyz, size):
    """xyz: float64 [N,3] (already shifted, all >= 0). Returns (keep_idx [M], inverse [N])."""
    g = np.floor(xyz / size).astype(np.int64)
    assert g.min() >= 0 and g.max() < 2**20, "coordinates outside the range the key packing supports"
    key = g[:, 0] | (g[:, 1] << 20) | (g[:, 2] << 40)
    d2 = ((xyz - (g + 0.5) * size) ** 2).sum(1)               # squared distance to the voxel centre
    order = np.lexsort((d2, key))                             # by voxel, then by distance to the centre
    key_sorted = key[order]
    first = np.ones(len(order), dtype=bool); first[1:] = key_sorted[1:] != key_sorted[:-1]
    keep_idx = order[first]                                   # closest point of every voxel
    voxel_id_sorted = np.cumsum(first) - 1
    inverse = np.empty(len(order), dtype=np.int64); inverse[order] = voxel_id_sorted
    return keep_idx, inverse.astype(np.int32)

# self-test on random points
_x = np.random.rand(20000, 3) * 0.5
_k, _inv = voxel_subsample(_x, 0.05)
assert len(_k) == len(np.unique(np.floor(_x / 0.05).astype(np.int64), axis=0))
assert np.all(np.floor(_x[_k][_inv] / 0.05) == np.floor(_x / 0.05)), "inverse map is wrong"
print("voxel_subsample self-test passed:", len(_x), "->", len(_k))

In [ ]:
stats = {"offset": OFFSET.tolist(), "voxels_m": VOXELS, "class_names": CLASS_NAMES, "stations": {}}

def process(station, split):
    path = os.path.join(RAW, "Gold" if split == "train" else "test_area", station + ".npy")
    a = np.load(path)
    xyz = a[:, :3] - OFFSET
    assert xyz.min() >= 0 and xyz.max() < 250, f"{station}: unexpected coordinate range {xyz.min(0)} .. {xyz.max(0)}"
    rgb = np.clip(a[:, 3:6], 0, 255).astype(np.uint8)
    inten = a[:, 6].astype(np.float32)
    label = a[:, 7].astype(np.uint8) if split == "train" else None      # manual labels
    pseudo = a[:, 8].astype(np.uint8) if split == "train" else None     # labels from the CAD model
    st = {"split": split, "points": int(len(a))}
    if label is not None:
        st["full"] = {int(c): int(n) for c, n in zip(*np.unique(label, return_counts=True))}
    for name, size in VOXELS.items():
        keep, inv = voxel_subsample(xyz, size)
        d = os.path.join(OUT, name); os.makedirs(d, exist_ok=True)
        out = {"xyz": xyz[keep].astype(np.float32), "rgb": rgb[keep], "intensity": inten[keep], "inverse": inv}
        if label is not None:
            out["label"], out["pseudo"] = label[keep], pseudo[keep]
            st[name] = {int(c): int(n) for c, n in zip(*np.unique(label[keep], return_counts=True))}
        np.savez_compressed(os.path.join(d, station + ".npz"), **out)
        st[name + "_points"] = int(len(keep))
    stats["stations"][station] = st
    print(f"{station} ({split}): {len(a):>10,} points -> " + ", ".join(f"{n}: {st[n + '_points']:,}" for n in VOXELS))
    return a if station == "S163" else None

t0 = time.time()
keep_for_check = None
for s in TRAIN:
    r = process(s, "train")
    if r is not None: keep_for_check = r
for s in TEST:
    process(s, "test")
with open(os.path.join(OUT, "stats.json"), "w") as fh:
    json.dump(stats, fh, indent=1)
print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# How much of each class survives, summed over the 7 labelled stations
tot = {k: np.zeros(18, dtype=np.int64) for k in ["full"] + list(VOXELS)}
for s in TRAIN:
    for k in tot:
        for c, n in stats["stations"][s][k].items(): tot[k][int(c)] += n
print(f"{'ID':>2} {'class':<14} {'full res':>12} {'2 cm':>10} {'1 cm':>10}")
for c in np.argsort(-tot["full"]):
    mark = "  <- small component" if c in SMALL else ""
    print(f"{c:>2} {CLASS_NAMES[c]:<14} {tot['full'][c]:>12,} {tot['2cm'][c]:>10,} {tot['1cm'][c]:>10,}{mark}")
print("\nTotals:", {k: int(v.sum()) for k, v in tot.items()})
size_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(OUT) for f in fs) / 1e6
print(f"Output size: {size_mb:.0f} MB in {OUT}")

## Is the class-ID → name mapping right? (T1.5)

The names were taken from a table image in the authors' repository; the files only contain numbers. Two independent checks on station S163:

- **Height above the floor per class.** If the mapping is right: Slab ≈ 0 m, Covering (ceiling) near the top, Damper (smoke detector) at ceiling height, Exit sign high on the wall, Switch around 1–1.3 m, Outlet low on the wall or at desk height, Lamp at the ceiling.
- **Pictures**: a side view coloured by class, and a top view of the small components.

In [ ]:
a = keep_for_check
z = a[:, 2]; lab = a[:, 7].astype(int)
floor = np.median(z[lab == 12]) if np.any(lab == 12) else z.min()
print(f"S163 floor level (median z of ID 12): {floor:.3f} m\n")
print(f"{'ID':>2} {'name (assumed)':<14} {'points':>10} {'height above floor: 5% / median / 95% (m)':>44}   {'thickness in z':>14}")
rows = {}
for c in np.unique(lab):
    h = z[lab == c] - floor
    q = np.percentile(h, [5, 50, 95])
    rows[int(c)] = [float(v) for v in q]
    print(f"{c:>2} {CLASS_NAMES[c]:<14} {len(h):>10,} {q[0]:>14.2f} / {q[1]:>5.2f} / {q[2]:>5.2f}   {q[2] - q[0]:>14.2f}")
with open(os.path.join(OUT, "S163_height_by_class.json"), "w") as fh:
    json.dump({"floor_z": float(floor), "height_5_50_95": rows}, fh, indent=1)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
sub = rng.choice(len(a), size=min(400_000, len(a)), replace=False)
cmap = matplotlib.colormaps["tab20"].resampled(18)

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
ax[0].scatter(a[sub, 0], a[sub, 2] - floor, c=lab[sub], cmap=cmap, vmin=0, vmax=17, s=0.2, linewidths=0)
ax[0].set_title("S163 - side view (x vs height), colour = class ID"); ax[0].set_xlabel("x (m)"); ax[0].set_ylabel("height above floor (m)")
handles = [plt.Line2D([], [], marker="o", ls="", color=cmap(c), label=f"{c} {CLASS_NAMES[c]}") for c in np.unique(lab)]
ax[0].legend(handles=handles, fontsize=7, ncol=2, loc="upper right", markerscale=1.5)

ax[1].scatter(a[sub, 0], a[sub, 1], c="lightgrey", s=0.1, linewidths=0)
for c in SMALL + [9]:
    m = lab == c
    if m.any(): ax[1].scatter(a[m, 0], a[m, 1], s=2, linewidths=0, label=f"{c} {CLASS_NAMES[c]} ({m.sum():,})")
ax[1].set_title("S163 - top view, small components and lamps"); ax[1].set_aspect("equal"); ax[1].legend(fontsize=8, markerscale=4)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "S163_class_check.png"), dpi=130); plt.show()

## After the run

1. Copy the class table and the height table into `docs/dataset_3dses.md`; state there whether the class mapping is confirmed.
2. Output tab → **New Dataset** → name `3dses-gold-v1`, private → add the other members as collaborators.
3. Record in `docs/experiment_log.md` as DATA-003 (notebook version, voxel sizes, offset `[900, 480, 100]`, point counts).

Loading one station later:

```python
d = np.load("/kaggle/input/3dses-gold-v1/2cm/S163.npz")
xyz, rgb, intensity, label = d["xyz"], d["rgb"], d["intensity"], d["label"]   # label absent for S170 / S171 / S180
full_res_prediction = prediction[d["inverse"]]                                 # back to every original point, original order
```